In [ ]:
!pip install sentence_transformers bm25s

In [ ]:
import pandas as pd
import numpy as np
import time
import warnings
import gc
import sys
import time
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer, CrossEncoder
import bm25s

warnings.filterwarnings('ignore')


train_path = '/kaggle/input/llm-agentic-legal-information-retrieval/train.csv'
val_path = '/kaggle/input/llm-agentic-legal-information-retrieval/val.csv'
test_path = '/kaggle/input/llm-agentic-legal-information-retrieval/test.csv'
laws_path = '/kaggle/input/llm-agentic-legal-information-retrieval/laws_de.csv'
court_path = '/kaggle/input/llm-agentic-legal-information-retrieval/court_considerations.csv'
output_path = 'submission.csv'

embedding_model_name = 'BAAI/bge-m3'
reranker_model_name = 'BAAI/bge-reranker-v2-m3'

use_gpu = torch.cuda.is_available()
num_gpus = torch.cuda.device_count() if use_gpu else 0

if num_gpus >= 2:
    device_emb = 'cuda:0'
    device_rerank = 'cuda:1'
elif num_gpus == 1:
    device_emb = 'cuda:0'
    device_rerank = 'cuda:0'
else:
    device_emb = 'cpu'
    device_rerank = 'cpu'

laws_batch_size = 8
query_batch_size = 512
max_length = 512
top_k_retrieval = 60
top_k_final = 10
text_truncate = 2000

use_hybrid = True
use_reranking = True
use_query_expansion = True


def load_data():
    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)
    test_df = pd.read_csv(test_path)
    laws_df = pd.read_csv(laws_path)
    laws_df['text'] = laws_df['text'].fillna('')
    court_df = pd.read_csv(court_path, usecols=['citation', 'text'])
    court_df['text'] = court_df['text'].fillna('')
    
    print(f"loaded {len(train_df)} train, {len(val_df)} val, {len(test_df)} test queries")
    print(f"loaded {len(laws_df)} laws, {len(court_df)} court decisions")
    
    return train_df, val_df, test_df, laws_df, court_df


def tokenize_for_bm25(texts):
    truncated = [str(t).lower()[:text_truncate] for t in texts]
    return bm25s.tokenize(truncated, stopwords='de')


def build_bm25_index(texts):
    tokens = tokenize_for_bm25(texts)
    retriever = bm25s.BM25()
    retriever.index(tokens)
    del tokens
    gc.collect()
    print("index ready")
    return retriever


def search_bm25(retriever, query, top_k):
    query_tokens = bm25s.tokenize([query.lower()], stopwords='de')
    docs, scores = retriever.retrieve(query_tokens, k=top_k)
    results = [(docs[0][i], scores[0][i]) for i in range(len(docs[0]))]
    return results


def load_embedding_model():
    print(f"loading embedding model on {device_emb}")
    model = SentenceTransformer(
        embedding_model_name,
        device=device_emb,
        model_kwargs={'torch_dtype': torch.float16}
    )
    print("model ready")
    return model


def encode_corpus(model, texts):
    truncated = [str(t)[:text_truncate] for t in texts]
    embeddings = model.encode(
        truncated,
        batch_size=laws_batch_size,
        convert_to_tensor=False,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    return embeddings


def encode_query(model, query):
    embedding = model.encode(
        query,
        convert_to_tensor=True,
        normalize_embeddings=True
    )
    return embedding.cpu().numpy()


def search_dense(query_embedding, corpus_embeddings, top_k):
    scores = np.dot(corpus_embeddings, query_embedding)
    top_indices = np.argsort(scores)[-top_k:][::-1]
    results = [(int(idx), float(scores[idx])) for idx in top_indices]
    return results


def load_reranker():
    print(f"loading reranker on {device_rerank}")
    model = CrossEncoder(
        reranker_model_name,
        device=device_rerank,
        max_length=512,
        automodel_args={'torch_dtype': torch.float16}
    )
    print("model ready")
    return model


def rerank_candidates(reranker, query, candidates):
    if not candidates:
        return []
    
    pairs = []
    citations = []
    
    for citation, text, _ in candidates:
        truncated = str(text)[:text_truncate]
        pairs.append([query, truncated])
        citations.append(citation)
    
    scores = reranker.predict(pairs, batch_size=8, show_progress_bar=False)
    scored = list(zip(citations, scores))
    scored.sort(key=lambda x: x[1], reverse=True)
    
    return [c[0] for c in scored]


def reciprocal_rank_fusion(sparse_results, dense_results, k=60):
    scores = {}
    
    for rank, (idx, _) in enumerate(sparse_results):
        scores[idx] = scores.get(idx, 0) + 1.0 / (k + rank)
    
    for rank, (idx, _) in enumerate(dense_results):
        scores[idx] = scores.get(idx, 0) + 1.0 / (k + rank)
    
    sorted_indices = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [idx for idx, _ in sorted_indices]


def hybrid_search_laws(bm25_laws, embedding_model, laws_embeddings, query, top_k):
    sparse_results = search_bm25(bm25_laws, query, top_k)
    query_embedding = encode_query(embedding_model, query)
    dense_results = search_dense(query_embedding, laws_embeddings, top_k)
    fused_indices = reciprocal_rank_fusion(sparse_results, dense_results)
    return fused_indices[:top_k]


def retrieve_citations(query, bm25_laws, bm25_courts, embedding_model, laws_embeddings, 
                       laws_df, court_df, reranker, law_citation_to_text, court_citation_to_text):
    
    law_indices = hybrid_search_laws(bm25_laws, embedding_model, laws_embeddings, query, top_k_retrieval)
    law_citations = [laws_df.iloc[idx]['citation'] for idx in law_indices if 0 <= idx < len(laws_df)]
    
    if use_query_expansion and law_citations:
        expanded_query = f"{query} {law_citations[0]}"
    else:
        expanded_query = query
    
    court_results = search_bm25(bm25_courts, expanded_query, top_k_retrieval)
    court_citations = [court_df.iloc[idx]['citation'] for idx, _ in court_results if 0 <= idx < len(court_df)]
    
    all_citations = []
    seen = set()
    for citation in law_citations + court_citations:
        if citation not in seen:
            all_citations.append(citation)
            seen.add(citation)
    
    if use_reranking and all_citations:
        candidates = []
        for citation in all_citations[:top_k_retrieval]:
            text = law_citation_to_text.get(citation) or court_citation_to_text.get(citation)
            if text:
                candidates.append((citation, text, 0.0))
        
        reranked = rerank_candidates(reranker, query, candidates)
        return reranked[:top_k_final]
    else:
        return all_citations[:top_k_final]


def main():
    start_time = time.time()
    
    print("loading data...")
    train_df, val_df, test_df, laws_df, court_df = load_data()
    
    print("building bm25 indices...")
    bm25_laws = build_bm25_index(laws_df['text'].tolist())
    bm25_courts = build_bm25_index(court_df['text'].tolist())
    
    embedding_model = load_embedding_model()
    
    print("encoding laws corpus...")
    laws_embeddings = encode_corpus(embedding_model, laws_df['text'].tolist())
    print("  encoding complete")
    
    reranker = load_reranker()
    
    law_citation_to_text = dict(zip(laws_df['citation'], laws_df['text']))
    court_citation_to_text = dict(zip(court_df['citation'], court_df['text']))
    
    print(f"processing {len(test_df)} test queries...")
    predictions = []
    
    for i, row in test_df.iterrows():
        citations = retrieve_citations(
            row['query'], bm25_laws, bm25_courts, embedding_model, laws_embeddings,
            laws_df, court_df, reranker, law_citation_to_text, court_citation_to_text
        )
        predictions.append(citations)
        
        if (i + 1) % 100 == 0:
            print(f"processed {i+1}/{len(test_df)}")
    
    submission_data = []
    for qid, citations in zip(test_df['query_id'], predictions):
        submission_data.append({
            'query_id': qid,
            'predicted_citations': ';'.join(citations)
        })
    
    submission_df = pd.DataFrame(submission_data)
    submission_df.to_csv(output_path, index=False)
    
    elapsed = time.time() - start_time
    print(f"completed in {elapsed/60:.1f} minutes")
    print(f"saved {len(submission_df)} predictions to {output_path}")
    
    return submission_df


if __name__ == "__main__":
    submission_df = main()